# Notebook 05 - Final Load and Tableau Prep
computing kpis and exporting aggregated csvs for the dashboard


In [1]:
import pandas as pd
import json
import os

df = pd.read_csv('../data/processed/nyc_parking_clean.csv')
print("loaded", len(df), "rows,", len(df.columns), "columns")


loaded 12000 rows, 38 columns


### computing kpis
these are the headline numbers that go at the top of our tableau dashboard


In [2]:
total_tickets = len(df)
total_revenue = float(df['Fine_Amount'].sum())
avg_fine = float(df['Fine_Amount'].mean().round(2))
oos_pct = float((df['Is_OutOfState'].mean() * 100).round(1))
repeat_pct = float((df['Is_Repeat_Offender'].mean() * 100).round(1))
safety_pct = float((df['Violation_Severity'] == 'Safety-Critical').mean() * 100)
avoidable_pct = float((df['Is_Avoidable'].mean() * 100).round(1))
unique_plates = int(df['Plate_ID'].nunique())
peak_month = str(df.groupby('Month_Name')['Summons_Number'].count().idxmax())
top_borough = str(df.groupby('Violation_County')['Fine_Amount'].sum().idxmax())
commercial_pct = float((df['Vehicle_Type_Group'] == 'Commercial').mean() * 100)
weekend_pct = float((df['Is_Weekend'].mean() * 100).round(1))
avg_car_age = float(df['Vehicle_Age'].mean().round(1))
top_violation = str(df['Violation_Description'].mode()[0])
oos_revenue = float(df[df['Is_OutOfState'] == 1]['Fine_Amount'].sum())

kpis = {
    'total_tickets': total_tickets,
    'total_revenue': total_revenue,
    'avg_fine_per_ticket': avg_fine,
    'outofstate_pct': oos_pct,
    'repeat_offender_pct': repeat_pct,
    'safety_violation_pct': round(safety_pct, 1),
    'avoidable_pct': avoidable_pct,
    'unique_vehicles': unique_plates,
    'peak_month': peak_month,
    'top_borough_revenue': top_borough,
    'commercial_vehicle_pct': round(commercial_pct, 1),
    'weekend_ticket_pct': weekend_pct,
    'avg_vehicle_age': avg_car_age,
    'most_frequent_violation': top_violation,
    'out_of_state_revenue': oos_revenue
}

with open('../docs/kpi_summary.json', 'w') as f:
    json.dump(kpis, f, indent=2)

print("15 kpis computed")
for k, v in kpis.items():
    print(f"  {k}: {v}")


15 kpis computed
  total_tickets: 12000
  total_revenue: 1031500.0
  avg_fine_per_ticket: 85.96
  outofstate_pct: 47.9
  repeat_offender_pct: 0.0
  safety_violation_pct: 22.6
  avoidable_pct: 56.9
  unique_vehicles: 11656
  peak_month: August
  top_borough_revenue: Manhattan
  commercial_vehicle_pct: 24.8
  weekend_ticket_pct: 16.0
  avg_vehicle_age: 8.0
  most_frequent_violation: No Parking - Street Cleaning
  out_of_state_revenue: 492475.0


### exporting 6 csv files for tableau
we pre-aggregate in python so tableau doesnt have to process 12000 rows every time


In [3]:
export_dir = '../data/processed/tableau_exports'
os.makedirs(export_dir, exist_ok=True)

borough = df.groupby('Violation_County').agg(
    ticket_count=('Summons_Number', 'count'),
    total_revenue=('Fine_Amount', 'sum'),
    avg_fine=('Fine_Amount', 'mean')
).reset_index()
borough['avg_fine'] = borough['avg_fine'].round(2)
borough.to_csv(f'{export_dir}/borough_summary.csv', index=False)
print("1. borough_summary.csv ->", len(borough), "rows")


1. borough_summary.csv -> 5 rows


In [4]:
month_order = ['January','February','March','April','May','June','July','August','September','October','November','December']
monthly = df.dropna(subset=['Month_Name']).groupby('Month_Name').agg(
    ticket_count=('Summons_Number', 'count'),
    total_revenue=('Fine_Amount', 'sum')
).reindex(month_order).reset_index()
monthly.to_csv(f'{export_dir}/monthly_trend.csv', index=False)
print("2. monthly_trend.csv ->", len(monthly), "rows")


2. monthly_trend.csv -> 12 rows


In [5]:
streets = df.groupby('Street_Name').agg(
    ticket_count=('Summons_Number', 'count'),
    total_revenue=('Fine_Amount', 'sum')
).reset_index().sort_values('ticket_count', ascending=False)
streets.to_csv(f'{export_dir}/streets_summary.csv', index=False)
print("3. streets_summary.csv ->", len(streets), "rows")


3. streets_summary.csv -> 17 rows


In [6]:
vehicle = df.groupby('Vehicle_Make').agg(
    ticket_count=('Summons_Number', 'count')
).reset_index().sort_values('ticket_count', ascending=False)
vehicle.to_csv(f'{export_dir}/vehicle_make_summary.csv', index=False)
print("4. vehicle_make_summary.csv ->", len(vehicle), "rows")


4. vehicle_make_summary.csv -> 16 rows


In [7]:
offender = df.dropna(subset=['Offender_Tier']).groupby('Offender_Tier').agg(
    ticket_count=('Summons_Number', 'count'),
    total_revenue=('Fine_Amount', 'sum')
).reset_index()
offender.to_csv(f'{export_dir}/offender_tier_summary.csv', index=False)
print("5. offender_tier_summary.csv ->", len(offender), "rows")


5. offender_tier_summary.csv -> 1 rows


In [8]:
state_group = df.groupby(['State_Group', 'Violation_County']).agg(
    ticket_count=('Summons_Number', 'count'),
    total_revenue=('Fine_Amount', 'sum')
).reset_index()
state_group.to_csv(f'{export_dir}/state_group_summary.csv', index=False)
print("6. state_group_summary.csv ->", len(state_group), "rows")


6. state_group_summary.csv -> 15 rows


### validation checks
making sure everything looks correct before we move to tableau


In [9]:
assert len(df) >= 5000, "not enough rows"
assert df['Fine_Amount'].isnull().sum() == 0, "fine amount still has nulls"
assert len(kpis) == 15, "missing kpis"

before_after = {
    "rows_before": 12000,
    "rows_after": len(df),
    "cols_before": 14,
    "cols_after": len(df.columns),
    "engineered_features": len(df.columns) - 14
}
with open('../docs/before_after_summary.json', 'w') as f:
    json.dump(before_after, f, indent=2)

print("all validations passed")
print(f"started with 12000 rows 14 cols -> ended with {len(df)} rows {len(df.columns)} cols")
print(f"engineered {len(df.columns) - 14} new features")
print("ready for tableau")


all validations passed
started with 12000 rows 14 cols -> ended with 12000 rows 38 cols
engineered 24 new features
ready for tableau
